In [ ]:
import xarray as xr
import cartopy.crs as ccrs
import matplotlib as mpl
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import numpy as np
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from datetime import datetime, UTC
import urllib
import shutil
import gzip
from herbie.toolbox import EasyMap, pc

In [96]:
s3 = boto3.client("s3", config=Config(signature_version=UNSIGNED))

cache = {
    "observations": None,
    "forecast_short": None,
    "surface_analysis_url": "https://www.wpc.ncep.noaa.gov/sfc/namussfc00wbg.gif",
    "radar_frames": [], 
    "satellite_vis_frames": [],
    "satellite_ir_frames": [],
}


def fetch_radar_frames(n_frames=10):
    """
    Fetch the most recent N MRMS reflectivity composite frames from AWS.
    Frames are stored as base64 strings so they can be sent as JSON to the browser.
    """
    bucket = "noaa-mrms-pds"
    date = datetime.now(UTC).strftime("%Y%m%d")
    prefix = f"CONUS/ReflectivityAtLowestAltitude_00.50/{date}"
    try:
        response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
        objects = sorted(
            response.get("Contents", []),
            key=lambda x: x["LastModified"],
            reverse=True
        )
        frames = []
        for obj in objects[:5*n_frames]:
            # MRMS files are gzip-compressed binary (GRIB2) — not directly renderable.
            # We'll handle rendering in a follow-up step; for now store the raw key list.
            if obj["LastModified"].minute%10 == 0:
                frames.append(obj["Key"])
        cache["radar_frames"] = frames
        print(f"[radar] Cached {len(frames)} frame keys")
    except Exception as e:
        print(f"[radar] Failed: {e}")

In [ ]:
def read_radar(data_file):
    try:
        out_path = data_file.replace(".gz", "")
        with gzip.open(data_file, "rb") as f_in:
            with open(out_path, "wb") as f_out:
                shutil.copyfileobj(f_in, f_out)
        data = xr.load_dataarray(out_path, engine="cfgrib", decode_timedelta=True)
    except Exception as e:
        print("Failed.")
    
    return data

In [ ]:
def make_plot():

    lat1, lon1, lat2, lon2 = (41,   -79,  44,  -74)

    width_in = 6 * (lon2 - lon1) / (lat2 - lat1)
    height_in = 6
    dpi = 150
    
    # Round pixel dimensions to nearest even number
    width_px = round(width_in * dpi / 2) * 2
    height_px = round(height_in * dpi / 2) * 2

    fig = plt.figure(figsize=[width_px/dpi,height_px/dpi], constrained_layout=True, dpi=dpi)
    ax = fig.add_subplot(projection=pc)

    ax = EasyMap("10m", add_coastlines=True,
                 coastlines_kw={"color":"#1b2433"}, ax=ax)
    ax = ax.LAND(facecolor="#5C636A", edgecolor="k", linewidth=1)
    ax = ax.BORDERS(color="#1b2433", linewidth=1, zorder=16)
    ax = ax.STATES(edgecolor="#1b2433", linewidth=1, zorder=15)
    ax = ax.COUNTIES(edgecolor="#1b2433", linewidth=0.5, zorder=15)
    ax = ax.LAKES(facecolor="#203251", linewidth=0.5, zorder=14)
    ax = ax.OCEAN(facecolor="#203251", linewidth=0.5, zorder=14)
    ax = ax.ax

    ax.set_extent([lon1, lon2, lat1, lat2], crs=pc)

    return fig, ax

In [55]:
"""
Radar (dBZ)
Useful for:
* Model Reflctivity
* MRMS
"""
colorlist1 = ["#383D4C00", "#9AA8D57C", "#5F79CFBF"] # 0-15
cmap1 = mpl.colors.LinearSegmentedColormap.from_list("radar1",colorlist1, N=15)
mpl.colormaps.register(cmap1, name="r1", force=True)
cm1 = plt.get_cmap("r1")(np.linspace(0,1,15))

colorlist2 = ["#7FD488", "#42BA32", "#37AB28", "#006D0B"] # 15-30
cmap2 = mpl.colors.LinearSegmentedColormap.from_list("radar2",colorlist2, N=15)
mpl.colormaps.register(cmap2, name="r2", force=True)
cm2 = plt.get_cmap("r2")(np.linspace(0,1,15))

colorlist3 = ["#FCF45E", "#AAAA00"] #30-40
cmap3 = mpl.colors.LinearSegmentedColormap.from_list("radar3",colorlist3, N=10)
mpl.colormaps.register(cmap3, name="r3", force=True)
cm3 = plt.get_cmap("r3")(np.linspace(0,1,10))

colorlistO = ["#FA933E", "#F95F00",] #40-50
cmapO = mpl.colors.LinearSegmentedColormap.from_list("radarO",colorlistO, N=10)
mpl.colormaps.register(cmapO, name="rO", force=True)
cmO = plt.get_cmap("rO")(np.linspace(0,1,10))

colorlist4 = ["#FF0000", "#960909"] #50-60
cmap4 = mpl.colors.LinearSegmentedColormap.from_list("radar4",colorlist4, N=10)
mpl.colormaps.register(cmap4, name="r4", force=True)
cm4 = plt.get_cmap("r4")(np.linspace(0,1,10))

colorlist5 = ["#F340BA", "#E088FD"] #60-70
cmap5 = mpl.colors.LinearSegmentedColormap.from_list("radar5",colorlist5, N=10)
mpl.colormaps.register(cmap5, name="r5", force=True)
cm5 = plt.get_cmap("r5")(np.linspace(0,1,10))

colors = np.concat([cm1, cm2, cm3, cmO, cm4, cm5])
cmap = mpl.colors.LinearSegmentedColormap.from_list('radar', colors)
cmap.set_under("#00000000")
cmap.set_over("#A31AFF")
cmap.set_bad("#00000000")

mpl.colormaps.register(cmap, name="radar", force=True)

In [73]:
for radar_frame in cache["radar_frames"]:
    data = read_radar(radar_frame)
    fig, ax = make_plot()
    p = ax.pcolormesh(data.longitude, data.latitude, data.values, cmap="radar", vmin=0, vmax=70, zorder=20)
    fig.canvas.draw()
    cax = inset_axes(ax, width="100%", height="5%",
                 loc="lower center",
                 bbox_to_anchor=(0, -0.08, 1, 1),
                 bbox_transform=ax.transAxes,
                 borderpad=0)

    cb = plt.colorbar(
    p,
    cax=cax,
    orientation="horizontal",
    spacing="proportional",
    ticks = np.arange(0, 70, 5), 
    )

    cb.ax.tick_params(color="#2a2724")
    cb.ax.set_xlabel("Reflectivity (dBZ)", color="#2a2724", size=18)

    fig.patch.set_facecolor("#e8e5de")

    ax.set_title(f"MRMS Radar Reflectivity Mosaic / {data.time.values.astype(str)[0:11]} {data.time.values.astype(str)[11:16]}Z / Upstate NY", size=18)

    plt.savefig(f"cache/radar/{data.time.values.astype(str)[0:16]}.png", dpi=150, bbox_inches="tight")
    plt.close()

ECCODES ERROR   :  Key dataTime (unpack_long): Truncating time: non-zero seconds(41) ignored
ECCODES ERROR   :  Key dataTime (unpack_long): Truncating time: non-zero seconds(41) ignored
/opt/anaconda3/envs/weather_plots/lib/python3.14/site-packages/cartopy/mpl/feature_artist.py:143: UserWarning: facecolor will have no effect as it has been defined as "never".
  warnings.warn('facecolor will have no effect as it has been '
ECCODES ERROR   :  Key dataTime (unpack_long): Truncating time: non-zero seconds(39) ignored
ECCODES ERROR   :  Key dataTime (unpack_long): Truncating time: non-zero seconds(39) ignored
/opt/anaconda3/envs/weather_plots/lib/python3.14/site-packages/cartopy/mpl/feature_artist.py:143: UserWarning: facecolor will have no effect as it has been defined as "never".
  warnings.warn('facecolor will have no effect as it has been '


Can't create file 'cache/radar/MRMS_ReflectivityAtLowestAltitude_00.50_20260422-031838.grib2.gz.5b7b6.idx'
Traceback (most recent call last):
  File "/opt/anaconda3/envs/weather_plots/lib/python3.14/site-packages/cfgrib/messages.py", line 274, in itervalues
    yield self.filestream.message_from_file(file, errors=errors)
          ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/weather_plots/lib/python3.14/site-packages/cfgrib/messages.py", line 341, in message_from_file
    return Message.from_file(file, offset, **kwargs)
           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/weather_plots/lib/python3.14/site-packages/cfgrib/messages.py", line 105, in from_file
    raise EOFError("End of file: %r" % file)
EOFError: End of file: <_io.BufferedReader name='cache/radar/MRMS_ReflectivityAtLowestAltitude_00.50_20260422-031838.grib2.gz'>

During handling of the above exception, another exception occurred:

Traceback (most recent ca

EOFError: No valid message found: 'cache/radar/MRMS_ReflectivityAtLowestAltitude_00.50_20260422-031838.grib2.gz'